# CDBurner10 Upscaler (.NET 10 + CUDA ONNX on Google Colab)

Run **CDBurner10.Upscaler.Cli** natively on Google Colab's **A100 / L4 / T4** GPU using **.NET 10** and **ONNX Runtime CUDA** (`ArtCNN_R16F96` / `ArtCNN_R8F64`), reading and saving directly to Google Drive.

> **Runtime:** Go to *Runtime → Change runtime type → Hardware accelerator → GPU (A100 / L4 / T4)*.

In [ ]:
# 1. Check GPU and install .NET 10 SDK + FFmpeg
!nvidia-smi
!curl -sSL https://dot.net/v1/dotnet-install.sh | bash /dev/stdin --channel 10.0
!ln -sf /root/.dotnet/dotnet /usr/local/bin/dotnet
!dotnet --version
!ffmpeg -version | head -n 1

In [ ]:
# 2. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. Clone CDBurner10 and build cross-platform CDBurner10.Upscaler.Cli
!rm -rf /content/CDBurner10
!git clone --depth 1 https://github.com/Stevekk11/CDBurner10.git /content/CDBurner10
!dotnet build /content/CDBurner10/src/CDBurner10.Upscaler.Cli/CDBurner10.Upscaler.Cli.csproj -c Release

In [ ]:
# 4. Run .NET 10 CUDA ArtCNN Upscaler on a movie from Google Drive
INPUT_VIDEO = "/content/drive/MyDrive/Filmy/Asterix/Asterix a Kleopatra.mp4"  #@param {type:"string"}
OUTPUT_FOLDER = "/content/drive/MyDrive/Filmy/Asterix/Upscaled"  #@param {type:"string"}
MODEL = "ArtCNN_R16F96"  #@param ["ArtCNN_R16F96", "ArtCNN_R8F64"]
FORMAT = "mp4"  #@param ["mp4", "mkv"]
CRF = 18  #@param {type:"integer"}
PRESET = "fast"  #@param ["ultrafast", "fast", "medium", "slow"]
TILE_SIZE = 1024  #@param {type:"integer"}

import subprocess, shlex
cmd = [
    "dotnet", "run", "--no-build", "-c", "Release",
    "--project", "/content/CDBurner10/src/CDBurner10.Upscaler.Cli/CDBurner10.Upscaler.Cli.csproj",
    "--",
    INPUT_VIDEO, OUTPUT_FOLDER,
    "--model", MODEL,
    "--format", FORMAT,
    "--crf", str(CRF),
    "--preset", PRESET,
    "--tile", str(TILE_SIZE)
]
print("Running:", " ".join(shlex.quote(c) for c in cmd))
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="")
proc.wait()
assert proc.returncode == 0, f"Upscaler exited with code {proc.returncode}"